# Task 03 Solution: LangChain Support Agent

## Setup

In [ ]:
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser, JsonOutputParser

OPENAI_API_KEY = "your-api-key-here"
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0, api_key=OPENAI_API_KEY)
print("✓ LLM initialized")


In [ ]:
import json, os

# In Jupyter, os.getcwd() returns the notebook's directory (learning/, tasks/, solutions/)
# Fixtures are one level up at ../fixtures/input/
fixtures = os.path.normpath(os.path.join(os.getcwd(), "..", "fixtures", "input"))

with open(os.path.join(fixtures, "tickets.json")) as f:
    tickets = json.load(f)

with open(os.path.join(fixtures, "knowledge_base.json")) as f:
    knowledge_base = json.load(f)

with open(os.path.join(fixtures, "test_questions.json")) as f:
    test_questions = json.load(f)

print(f"✓ Loaded {len(tickets)} tickets, {len(knowledge_base)} KB articles, {len(test_questions)} test questions")


## Part 1: Define Tools

In [ ]:
import json
from langchain_core.tools import tool
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

# Build vector store for KB search
texts = [a["content"] for a in knowledge_base]
metadatas = [{"id": a["id"], "title": a["title"]} for a in knowledge_base]
embeddings_model = OpenAIEmbeddings(model="text-embedding-3-small", api_key=OPENAI_API_KEY)
vectorstore = FAISS.from_texts(texts, embeddings_model, metadatas=metadatas)
retriever = vectorstore.as_retriever(search_kwargs={"k": 3})

print("✓ Vector store ready")


In [ ]:
@tool
def classify_ticket(ticket_text: str) -> str:
    """Classify a customer support ticket by category and priority.
    Use this FIRST for every customer query to determine the type and urgency
    of the issue before deciding how to respond.

    Args:
        ticket_text: The customer's support message.

    Returns:
        JSON string with 'category' (billing/technical/account/shipping)
        and 'priority' (high/medium/low).
    """
    classify_prompt = ChatPromptTemplate.from_messages([
        ("system", 'Return ONLY valid JSON: {{"category": "billing|technical|account|shipping", "priority": "high|medium|low"}}'),
        ("human", "{ticket}"),
    ])
    chain = classify_prompt | llm | JsonOutputParser()
    try:
        result = chain.invoke({"ticket": ticket_text})
        return json.dumps(result)
    except Exception as e:
        return json.dumps({"category": "technical", "priority": "medium", "error": str(e)})


@tool
def search_knowledge_base(query: str) -> str:
    """Search the company knowledge base for policy and troubleshooting information.
    Use this when the customer asks about: refunds, billing, API rate limits, password
    reset, shipping tracking, account security, mobile app issues, subscription
    cancellation, team management, or any company procedures and policies.

    Args:
        query: The customer's question or the topic to search for.

    Returns:
        Relevant knowledge base articles with titles and content.
    """
    docs = retriever.invoke(query)
    if not docs:
        return "No relevant articles found in the knowledge base."
    return "\n\n".join(
        f"[{doc.metadata['title']}]\n{doc.page_content}"
        for doc in docs
    )


@tool
def escalate_to_human(reason: str, priority: str) -> str:
    """Escalate the support ticket to a human agent.
    Use this when: (1) the issue is high priority involving account compromise,
    data loss, or financial damage; (2) the knowledge base does not contain
    a resolution; (3) the customer explicitly requests to speak with a human.

    Args:
        reason: Brief description of why escalation is needed.
        priority: Urgency level: high, medium, or low.

    Returns:
        Confirmation that a human agent has been assigned.
    """
    wait_time = "1 hour" if priority == "high" else "4 hours" if priority == "medium" else "24 hours"
    return (
        f"✓ Ticket escalated to a human support agent (priority={priority}). "
        f"A team member will contact you within {wait_time}. "
        f"Reason: {reason}"
    )

tools = [classify_ticket, search_knowledge_base, escalate_to_human]
print(f"✓ Defined {len(tools)} tools")


In [ ]:
# Structural checks
for t in tools:
    assert hasattr(t, 'name')
    assert len(t.description) > 50
    print(f"  ✓ {t.name}")
print("✓ Tool checks passed")


In [ ]:
# Callable checks
r = classify_ticket.invoke({"ticket_text": "I was charged twice"})
parsed = json.loads(r)
assert "category" in parsed and "priority" in parsed
print(f"✓ classify_ticket: {parsed}")

kb = search_knowledge_base.invoke({"query": "refund policy"})
assert len(kb) > 50
print(f"✓ search_knowledge_base: {kb[:80]}...")

esc = escalate_to_human.invoke({"reason": "account compromise", "priority": "high"})
print(f"✓ escalate_to_human: {esc}")


## Part 2: Build Agent

In [ ]:
from langchain.agents import create_agent

agent = create_agent(
    llm,
    tools,
    system_prompt="""\
You are a customer support agent for a SaaS company. Follow this process for every query:

1. CLASSIFY: Use classify_ticket to determine the issue type and urgency
2. SEARCH: Use search_knowledge_base to find relevant policy or troubleshooting info
3. RESPOND: Give a clear, specific answer based on what you found
4. ESCALATE: If the issue is high priority (account compromise, data loss, financial damage)
   OR you cannot find a resolution, use escalate_to_human

Always be helpful, specific, and cite the relevant policy or steps from the knowledge base.""",
)
print("✓ Agent ready")


In [ ]:
assert hasattr(agent, 'invoke')
test_result = agent.invoke({"messages": [{"role": "user", "content": "Hello"}]})
assert "messages" in test_result
print("✓ Agent structure correct")


## Part 3: Run Agent

In [ ]:
def ask_agent(agent, question):
    result = agent.invoke({"messages": [{"role": "user", "content": question}]})
    return result["messages"][-1].content

answer1 = ask_agent(agent, "Why is my API returning 429 errors? How do I fix this?")
print("Answer:", answer1)


In [ ]:
assert len(answer1) > 50
assert any(kw in answer1.lower() for kw in ["429", "rate limit", "retry", "too many"])
print("✓ API rate limit answer correct")


In [ ]:
answer2 = ask_agent(agent, "My account was hacked! I see login attempts from Russia. URGENT!")
print("Answer:", answer2)


In [ ]:
assert len(answer2) > 30
handled = any(kw in answer2.lower() for kw in ["high", "escalat", "lock", "password", "2fa", "security"])
assert handled
print("✓ Urgent issue handled")


In [ ]:
answer3 = ask_agent(agent, "What happens to my data if I cancel my subscription?")
print("Answer:", answer3)


In [ ]:
assert any(kw in answer3.lower() for kw in ["90", "export", "delet", "cancel"])
print("✓ Cancellation answer correct")


## Part 4: Inspect Tool Calls

In [ ]:
from langchain_core.messages import ToolMessage

result = agent.invoke({
    "messages": [{"role": "user", "content": "How do I reset my password if my reset email is not arriving?"}]
})

tool_messages = [m for m in result["messages"] if isinstance(m, ToolMessage)]
tool_names_used = [m.name for m in tool_messages]
print(f"Tool calls: {tool_names_used}")
print(f"Answer: {result['messages'][-1].content}")

assert len(tool_messages) > 0
assert "search_knowledge_base" in tool_names_used or "classify_ticket" in tool_names_used
print("✓ Agent used tools correctly")
